# D2-07 Case study - Regionalised hydrogen pathways

This final Day 2 case study combines Brightway import work, foreground relinking, and regionalised LCIA with `edges`.

We will:
- import the bundled hydrogen workbook into Brightway
- relink the direct electricity demand of the electrolyser to the Spanish low-voltage mix
- compare **PEM**, **AEC**, and **SOEC** hydrogen production
- run the comparison across **all ImpactWorld+ 2.1 damage-oriented methods**

Each step includes a worked solution in a collapsed code cell.


## Learning goals

After this notebook, you should be able to:
- turn a bundled foreground workbook into an import-ready Brightway database
- relink foreground electricity exchanges to a country-specific background dataset
- run a multi-method `edges` comparison across several technologies
- interpret when the technology ranking stays stable, and when it changes across damage categories


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Bulle, C., Margni, M., Patouillard, L., Boulay, A.-M., Bourgault, G., De Bruille, V., Cao, V., Hauschild, M., Henderson, A., Humbert, S., Kashef-Haghighi, S., Kounina, A., Laurent, A., Levasseur, A., Liard, G., Rosenbaum, R. K., Roy, P.-O., Shaked, S., & Jolliet, O. (2019). IMPACT World+: a globally regionalized life cycle impact assessment method. *The International Journal of Life Cycle Assessment, 24*, 1653-1674. https://doi.org/10.1007/s11367-019-01583-0
- The foreground workbook in `tutorials/DAY 2/assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx` contains PEM, AEC, and SOEC hydrogen routes originally prepared for ecoinvent 3.10.


## 1) Import the hydrogen workbook

We keep the import workflow compact here, like in Day 1: create the importer, apply the standard strategies, match the background databases, and write the foreground database.

Tasks:
- load the bundled workbook from the Day 2 assets
- import and match the foreground against `bafu` and the biosphere database
- rename the imported hydrogen production datasets to the Spanish location `ES`
- write the foreground database
- confirm that the PEM, AEC, and SOEC activities were imported


In [ ]:
from pathlib import Path
import contextlib
import io
import json

import bw2data as bd
import bw2io as bi
import edges
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from edges import EdgeLCIA

foreground_path = Path("tutorials/DAY 2/assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx")
if not foreground_path.exists():
    foreground_path = Path("assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx")


bd.projects.set_current("barcelona-rlcia-2026")
importer = bi.ExcelImporter(foreground_path)
with contextlib.redirect_stdout(io.StringIO()):
    importer.apply_strategies()

hydrogen_db_name = importer.db_name
if hydrogen_db_name in bd.databases:
    del bd.databases[hydrogen_db_name]

bafu_db_name = next(name for name in sorted(bd.databases) if name.startswith("bafu"))
biosphere_db_name = next(name for name in sorted(bd.databases) if "biosphere" in name)

for dataset in importer.data:
    if "hydrogen production" in dataset.get("name", "").lower():
        dataset["location"] = "ES"
        for exc in dataset.get("exchanges", []):
            if exc.get("type") == "production":
                exc["location"] = "ES"

importer.match_database(
    fields=("name", "reference product", "location", "unit"),
)
importer.match_database(
    bafu_db_name,
    fields=("name", "reference product", "location", "unit"),
)
importer.match_database(
    biosphere_db_name,
    kind="biosphere",
    fields=("name", "categories", "unit"),
)

import_stats = importer.statistics()
print("Importer statistics before write:", import_stats)

with contextlib.redirect_stdout(io.StringIO()):
    importer.write_database()

hydrogen_db = bd.Database(hydrogen_db_name)
imported_hydrogen_table = pd.DataFrame(
    [
        {
            "name": act["name"],
            "reference product": act.get("reference product"),
            "location": act.get("location"),
        }
        for act in hydrogen_db
        if "hydrogen production" in act["name"].lower()
    ]
).sort_values("name")

print("Foreground path:", foreground_path.resolve())
display(imported_hydrogen_table)


## 2) Relink the direct electrolyser electricity demand to Spain

We now relink the **direct electricity use** of the three hydrogen production datasets to the Spanish low-voltage grid mix in `bafu`.

Tasks:
- identify the three foreground activities: PEM, AEC, and SOEC
- find the Spanish `Electricity, low voltage, at grid` dataset
- replace the direct electricity exchange in each hydrogen production activity
- verify that all three activities now point to `ES`


In [ ]:
def get_one(database_name, predicate):
    hits = [act for act in bd.Database(database_name) if predicate(act)]
    if len(hits) != 1:
        raise ValueError(f"Expected one hit in {database_name}, got {len(hits)}")
    return hits[0]


def relink_direct_electricity(activity, new_supplier):
    for exchange in list(activity.technosphere()):
        if "electricity" not in exchange.input["name"].lower():
            continue
        if exchange.input.get("unit") != "kilowatt hour":
            continue
        if exchange.input.key == new_supplier.key:
            continue

        payload = exchange.as_dict()
        exchange.delete()
        new_exchange = activity.new_exchange(
            input=new_supplier,
            amount=payload["amount"],
            type=payload["type"],
        )
        for key, value in payload.items():
            if key in {"amount", "type", "input", "output"}:
                continue
            new_exchange[key] = value
        new_exchange.save()


spanish_electricity = get_one(
    bafu_db_name,
    lambda act: act["name"] == "Electricity, low voltage, at grid" and act.get("location") == "ES",
)

hydrogen_activities = {
    "AEC": get_one(
        hydrogen_db_name,
        lambda act: "hydrogen production" in act["name"].lower()
        and "AEC" in act["name"]
        and "with steam" not in act["name"],
    ),
    "PEM": get_one(
        hydrogen_db_name,
        lambda act: "hydrogen production" in act["name"].lower()
        and "PEM" in act["name"],
    ),
    "SOEC": get_one(
        hydrogen_db_name,
        lambda act: "hydrogen production" in act["name"].lower()
        and "from SOEC electrolysis, from grid electricity" in act["name"]
        and "with steam" not in act["name"],
    ),
}

for activity in hydrogen_activities.values():
    relink_direct_electricity(activity, spanish_electricity)

relink_check = []
for technology, activity in hydrogen_activities.items():
    for exchange in activity.technosphere():
        if "electricity" in exchange.input["name"].lower():
            relink_check.append(
                {
                    "technology": technology,
                    "electricity amount": exchange.amount,
                    "supplier": exchange.input["name"],
                    "supplier location": exchange.input.get("location"),
                }
            )

display(pd.DataFrame(relink_check).sort_values("technology"))


## 3) Collect all ImpactWorld+ 2.1 damage-oriented methods

Tasks:
- ask `edges` which methods are available
- keep only the ImpactWorld+ methods tagged as `damage`
- create a tidy lookup table with method name, short label, and unit


In [ ]:
edges_data_dir = Path(edges.__file__).resolve().parent / "data"
impactworld_damage_methods = [
    method
    for method in edges.get_available_methods()
    if method[0] == "ImpactWorld+ 2.1" and method[-1] == "damage"
]

impactworld_damage_table = pd.DataFrame(
    [
        {
            "method": method,
            "method_name": json.loads(
                (edges_data_dir / f"{'_'.join(method)}.json").read_text(encoding="utf-8")
            )["name"],
            "short_label": method[1],
            "unit": json.loads(
                (edges_data_dir / f"{'_'.join(method)}.json").read_text(encoding="utf-8")
            ).get("unit"),
        }
        for method in impactworld_damage_methods
    ]
)

print("Damage-oriented ImpactWorld+ methods:", len(impactworld_damage_table))
display(impactworld_damage_table[["short_label", "unit"]])


## 4) Run `edges` for PEM, AEC, and SOEC across all ImpactWorld+ damage methods

This is the main comparison loop. We use:
- one `EdgeLCIA` run per method for the first technology
- `redo_lcia()` for the two other technologies

That keeps the method-level comparison compact while avoiding repeated setup work. We also save the location-level contributions for each run so we can build richer summary plots afterward.

This step may take a minute or two.


In [ ]:
hydrogen_results = []
hydrogen_location_results = []

for row in impactworld_damage_table.itertuples(index=False):
    lca = None
    with contextlib.redirect_stdout(io.StringIO()):
        for index, (technology, activity) in enumerate(hydrogen_activities.items()):
            if index == 0:
                lca = EdgeLCIA(demand={activity: 1}, method=row.method)
                lca.lci()
                lca.apply_strategies()
                lca.evaluate_cfs()
                lca.lcia()
            else:
                lca.redo_lcia({activity.id: 1})

            hydrogen_results.append(
                {
                    "technology": technology,
                    "method": row.short_label,
                    "method_name": row.method_name,
                    "unit": row.unit,
                    "score": float(lca.score),
                }
            )

            cf_table = lca.generate_cf_table().copy()
            if cf_table.empty or not {"consumer location", "impact"}.issubset(cf_table.columns):
                continue
            location_impacts = (
                cf_table.assign(
                    consumer_location=lambda df: df["consumer location"].fillna("Unknown")
                )
                .groupby("consumer_location", as_index=False)["impact"]
                .sum()
            )
            for location_row in location_impacts.itertuples(index=False):
                hydrogen_location_results.append(
                    {
                        "technology": technology,
                        "method": row.short_label,
                        "unit": row.unit,
                        "consumer_location": location_row.consumer_location,
                        "impact": float(location_row.impact),
                    }
                )

hydrogen_results_df = pd.DataFrame(hydrogen_results)
hydrogen_location_results_df = pd.DataFrame(hydrogen_location_results)
display(hydrogen_results_df.head())
print("Rows in result table:", len(hydrogen_results_df))


## 5) Summarize the technology ranking

The damage-oriented ImpactWorld+ methods split naturally into two comparable families:
- **human health** methods, reported in `DALY`
- **ecosystem quality** methods, reported in `PDF.m2.yr`

We therefore compare the technologies **within each unit family** using:
- stacked bars, where each electrolyser score is decomposed by indicator
- a 3D location-indicator plot, where color shows the technology and height shows the location-specific impact contribution


In [ ]:
from textwrap import fill


score_wide = hydrogen_results_df.pivot(index="method", columns="technology", values="score").sort_index()
method_units = impactworld_damage_table.set_index("short_label")["unit"].reindex(score_wide.index)

group_specs = [
    ("Human health (DALY)", "DALY"),
    ("Ecosystem quality (PDF.m2.yr)", "PDF.m2.yr"),
]

score_groups = {
    label: score_wide.loc[method_units == unit].copy()
    for label, unit in group_specs
}
technology_order = list(hydrogen_activities)
technology_palette = {
    "AEC": "#264653",
    "PEM": "#2A9D8F",
    "SOEC": "#E76F51",
}
technology_offsets = {"AEC": -0.22, "PEM": 0.0, "SOEC": 0.22}
technology_markers = {"AEC": "o", "PEM": "^", "SOEC": "s"}


def plot_indicator_stacks(ax, score_df, unit, title):
    if score_df.empty:
        ax.set_title(title, fontsize=12, pad=10)
        ax.text(0.5, 0.5, "No matched indicators", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    indicator_totals = score_df.abs().sum(axis=1).sort_values(ascending=False)
    top_indicators = list(indicator_totals.head(8).index)
    stacked = score_df.loc[top_indicators].T.reindex(technology_order)
    if len(indicator_totals) > len(top_indicators):
        stacked["Other"] = score_df.drop(index=top_indicators).sum(axis=0).reindex(technology_order)
    indicator_labels = [fill(label, 16) for label in stacked.columns]
    colors = plt.cm.tab20(np.linspace(0, 1, len(stacked.columns)))
    bottom = np.zeros(len(stacked.index))
    for color, column, label in zip(colors, stacked.columns, indicator_labels):
        values = stacked[column].to_numpy(dtype=float)
        ax.bar(
            stacked.index,
            values,
            bottom=bottom,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            label=label,
        )
        bottom += values
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel("Hydrogen technology")
    ax.set_ylabel(unit)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=0)


def magnitude_log10(values):
    values = np.asarray(values, dtype=float)
    return np.log10(1 + np.abs(values))


def plot_location_indicator_3d(ax, location_df, unit, title):
    if location_df.empty:
        ax.set_title(title, fontsize=12, pad=10)
        ax.text2D(0.5, 0.5, "No location contributions available", ha="center", va="center", transform=ax.transAxes)
        return

    top_methods = (
        location_df.groupby("method")["impact"]
        .apply(lambda s: s.abs().sum())
        .sort_values(ascending=False)
        .head(5)
        .index
    )
    top_locations = (
        location_df.groupby("consumer_location")["impact"]
        .apply(lambda s: s.abs().sum())
        .sort_values(ascending=False)
        .head(5)
        .index
    )
    plot_df = location_df[
        location_df["consumer_location"].isin(top_locations)
        & location_df["method"].isin(top_methods)
    ].copy()
    method_order = (
        plot_df.groupby("method")["impact"]
        .apply(lambda s: s.abs().sum())
        .sort_values(ascending=False)
        .index
    )
    location_order = list(top_locations)
    if len(method_order) == 0 or len(location_order) == 0:
        ax.set_title(title, fontsize=12, pad=10)
        ax.text2D(0.5, 0.5, "No location contributions available", ha="center", va="center", transform=ax.transAxes)
        return
    x_positions = np.arange(len(method_order), dtype=float) * 2.0
    y_positions = np.arange(len(location_order), dtype=float)
    x_map = {method: x_positions[idx] for idx, method in enumerate(method_order)}
    y_map = {location: y_positions[idx] for idx, location in enumerate(location_order)}
    indicator_colors = plt.cm.Set2(np.linspace(0, 1, len(method_order)))
    location_colors = plt.cm.Set3(np.linspace(0, 1, len(location_order)))

    for technology in technology_order:
        tech_df = plot_df[plot_df["technology"] == technology]
        if tech_df.empty:
            continue
        xs = [x_map[m] + technology_offsets[technology] for m in tech_df["method"]]
        ys = [y_map[l] for l in tech_df["consumer_location"]]
        zs_raw = tech_df["impact"].to_numpy(dtype=float)
        zs = magnitude_log10(zs_raw)
        for x_val, y_val, z_val in zip(xs, ys, zs):
            ax.plot(
                [x_val, x_val],
                [y_val, y_val],
                [0, z_val],
                color=technology_palette[technology],
                alpha=0.28,
                linewidth=1.1,
            )
        ax.scatter(
            xs,
            ys,
            zs,
            color=technology_palette[technology],
            marker=technology_markers[technology],
            s=60,
            alpha=0.95,
            label=technology,
            depthshade=False,
            edgecolors="black",
            linewidths=0.35,
        )

    x_min = x_positions[0] - 0.45
    x_max = x_positions[-1] + 0.45
    y_min = 0.0
    y_max = y_positions[-1] if len(y_positions) > 1 else 1.0
    for xpos, color in zip(x_positions, indicator_colors):
        ax.plot(
            [xpos, xpos],
            [y_min, y_max],
            [0, 0],
            color=color,
            alpha=0.45,
            linewidth=1.4,
        )
    for ypos, color in zip(y_positions, location_colors):
        ax.plot(
            [x_min, x_max],
            [ypos, ypos],
            [0, 0],
            color=color,
            alpha=0.45,
            linewidth=1.2,
        )

    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel("Indicator")
    ax.set_ylabel("Location")
    ax.set_zlabel(f"log10(1 + |score|) [{unit}]")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0, y_max + 0.1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(list(method_order), rotation=24, ha="right", fontsize=8)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(location_order, fontsize=8)
    z_extent = max(
        np.nanmax(magnitude_log10(plot_df["impact"].to_numpy(dtype=float))),
        1e-6,
    )
    z_ticks = np.linspace(0, z_extent, 5)
    ax.set_zticks(z_ticks)
    ax.set_zticklabels([f"{tick:.1f}" for tick in z_ticks], fontsize=8)
    ax.set_zlim(0, z_extent * 1.05)
    for tick, color in zip(ax.get_xticklabels(), indicator_colors):
        tick.set_color(color)
    for tick, color in zip(ax.get_yticklabels(), location_colors):
        tick.set_color(color)
    ax.xaxis.pane.set_alpha(0.05)
    ax.yaxis.pane.set_alpha(0.05)
    ax.zaxis.pane.set_alpha(0.0)
    ax.grid(True, alpha=0.25)
    ax.view_init(elev=26, azim=-56)


location_groups = {
    label: hydrogen_location_results_df.loc[hydrogen_location_results_df["unit"] == unit].copy()
    for label, unit in group_specs
}

display(score_groups["Human health (DALY)"].round(6))
display(score_groups["Ecosystem quality (PDF.m2.yr)"].round(6))

fig = plt.figure(figsize=(20, 13), constrained_layout=True)
gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[0.95, 1.25],
    width_ratios=[1.0, 0.34, 1.0, 0.34],
)

ax_hh_stack = fig.add_subplot(gs[0, 0])
ax_hh_legend = fig.add_subplot(gs[0, 1])
ax_eco_stack = fig.add_subplot(gs[0, 2])
ax_eco_legend = fig.add_subplot(gs[0, 3])
ax_hh_3d = fig.add_subplot(gs[1, 0:2], projection="3d")
ax_eco_3d = fig.add_subplot(gs[1, 2:4], projection="3d")

plot_indicator_stacks(
    ax_hh_stack,
    score_groups["Human health (DALY)"],
    "DALY",
    "A) Human health scores by indicator",
)
plot_indicator_stacks(
    ax_eco_stack,
    score_groups["Ecosystem quality (PDF.m2.yr)"],
    "PDF.m2.yr",
    "B) Ecosystem scores by indicator",
)

hh_handles, hh_labels = ax_hh_stack.get_legend_handles_labels()
eco_handles, eco_labels = ax_eco_stack.get_legend_handles_labels()
ax_hh_legend.axis("off")
ax_eco_legend.axis("off")
ax_hh_legend.legend(
    hh_handles,
    hh_labels,
    title="Indicators",
    loc="center left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
)
ax_eco_legend.legend(
    eco_handles,
    eco_labels,
    title="Indicators",
    loc="center left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
)

plot_location_indicator_3d(
    ax_hh_3d,
    location_groups["Human health (DALY)"],
    "DALY",
    "C) Human health location-indicator contributions",
)
plot_location_indicator_3d(
    ax_eco_3d,
    location_groups["Ecosystem quality (PDF.m2.yr)"],
    "PDF.m2.yr",
    "D) Ecosystem location-indicator contributions",
)
ax_hh_3d.legend(loc="upper left", frameon=False)
ax_eco_3d.legend(loc="upper left", frameon=False)

fig.suptitle(
    "PEM, AEC, and SOEC across ImpactWorld+ damage endpoints",
    fontsize=16,
    y=1.02,
)
plt.show()


## 6) Interpretation questions

Use the split-by-unit score tables and the four-panel figure to discuss:
- Does the same electrolyser come out best for both **human health** and **ecosystem quality**?
- Within each unit family, which indicators contribute most strongly to the stacked totals of PEM, AEC, and SOEC?
- In the 3D plots, which consumer locations dominate the indicator-level contributions?
- After forcing all three technologies onto the **same Spanish electricity background**, which remaining differences still come from the foreground inventories themselves?


## Recap

After this notebook, you should now be able to:
- import and relink a hydrogen foreground against `bafu`
- use `edges.get_available_methods()` to select a family of regionalised methods
- redirect the direct electricity demand of PEM, AEC, and SOEC to Spain
- run `edges` across all ImpactWorld+ damage methods
- compare human-health and ecosystem endpoints separately, and interpret their location-specific contributions
